In [1]:
# ============================================================
# Step 22 - Load Clean Ultrasound Training Data
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import torch

from torch.utils.data import Dataset, DataLoader
from PIL import Image

# ------------------------------------------------------------
# Project root
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

ULTRASOUND_PROCESSED_DIR = (
    PROJECT_ROOT
    / "datasets"
    / "processed"
    / "ultrasound"
)

# ------------------------------------------------------------
# Clean split manifests
# ------------------------------------------------------------

TRAIN_PATH = (
    ULTRASOUND_PROCESSED_DIR
    / "clean_train_manifest.csv"
)

VAL_PATH = (
    ULTRASOUND_PROCESSED_DIR
    / "clean_validation_manifest.csv"
)

TEST_PATH = (
    ULTRASOUND_PROCESSED_DIR
    / "clean_test_manifest.csv"
)

# ------------------------------------------------------------
# Load manifests
# ------------------------------------------------------------

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Clean manifests loaded successfully.")

print("\nTraining images:", len(train_df))
print("Validation images:", len(val_df))
print("Testing images:", len(test_df))

print("\nTraining cases:", train_df["case_number"].nunique())
print("Validation cases:", val_df["case_number"].nunique())
print("Testing cases:", test_df["case_number"].nunique())


# ------------------------------------------------------------
# PyTorch Dataset
# ------------------------------------------------------------

class UltrasoundSegmentationDataset(Dataset):

    def __init__(self, dataframe):
        self.dataframe = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        image = Image.open(
            row["processed_image_path"]
        ).convert("RGB")

        mask = Image.open(
            row["processed_mask_path"]
        ).convert("L")

        image_array = np.asarray(
            image,
            dtype=np.float32
        ) / 255.0

        mask_array = np.asarray(
            mask,
            dtype=np.float32
        )

        mask_array = (
            mask_array > 127
        ).astype(np.float32)

        image_tensor = torch.tensor(
            image_array,
            dtype=torch.float32
        ).permute(2, 0, 1)

        mask_tensor = torch.tensor(
            mask_array,
            dtype=torch.float32
        ).unsqueeze(0)

        return image_tensor, mask_tensor


# ------------------------------------------------------------
# Create datasets
# ------------------------------------------------------------

train_dataset = UltrasoundSegmentationDataset(train_df)
val_dataset = UltrasoundSegmentationDataset(val_df)
test_dataset = UltrasoundSegmentationDataset(test_df)


# ------------------------------------------------------------
# DataLoaders
# ------------------------------------------------------------

BATCH_SIZE = 8

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)


# ------------------------------------------------------------
# Verify one batch
# ------------------------------------------------------------

images, masks = next(
    iter(train_loader)
)

print("\nDataLoader verification:")
print("Image batch shape:", images.shape)
print("Mask batch shape:", masks.shape)
print("Image range:", float(images.min()), "to", float(images.max()))
print("Mask values:", torch.unique(masks))

Clean manifests loaded successfully.

Training images: 367
Validation images: 45
Testing images: 43

Training cases: 299
Validation cases: 37
Testing cases: 38

DataLoader verification:
Image batch shape: torch.Size([8, 3, 224, 224])
Mask batch shape: torch.Size([8, 1, 224, 224])
Image range: 0.0 to 1.0
Mask values: tensor([0., 1.])


In [2]:
# ============================================================
# Step 23 - Define Lightweight U-Net
# ============================================================

import torch
import torch.nn as nn


# ------------------------------------------------------------
# Double Convolution Block
# ------------------------------------------------------------

class DoubleConv(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


# ------------------------------------------------------------
# Lightweight U-Net
# ------------------------------------------------------------

class UNetSmall(nn.Module):

    def __init__(self, in_channels=3, out_channels=1):
        super().__init__()

        # Encoder
        self.enc1 = DoubleConv(in_channels, 16)
        self.enc2 = DoubleConv(16, 32)
        self.enc3 = DoubleConv(32, 64)

        self.pool = nn.MaxPool2d(2)

        # Bottleneck
        self.bottleneck = DoubleConv(64, 128)

        # Decoder
        self.up3 = nn.ConvTranspose2d(
            128,
            64,
            kernel_size=2,
            stride=2
        )
        self.dec3 = DoubleConv(128, 64)

        self.up2 = nn.ConvTranspose2d(
            64,
            32,
            kernel_size=2,
            stride=2
        )
        self.dec2 = DoubleConv(64, 32)

        self.up1 = nn.ConvTranspose2d(
            32,
            16,
            kernel_size=2,
            stride=2
        )
        self.dec1 = DoubleConv(32, 16)

        # Final segmentation output
        self.output = nn.Conv2d(
            16,
            out_channels,
            kernel_size=1
        )

    def forward(self, x):

        # Encoder
        e1 = self.enc1(x)
        p1 = self.pool(e1)

        e2 = self.enc2(p1)
        p2 = self.pool(e2)

        e3 = self.enc3(p2)
        p3 = self.pool(e3)

        # Bottleneck
        b = self.bottleneck(p3)

        # Decoder
        d3 = self.up3(b)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        return self.output(d1)


# ------------------------------------------------------------
# Create model
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

segmentation_model = UNetSmall(
    in_channels=3,
    out_channels=1
).to(device)


# ------------------------------------------------------------
# Model verification
# ------------------------------------------------------------

total_parameters = sum(
    p.numel()
    for p in segmentation_model.parameters()
)

trainable_parameters = sum(
    p.numel()
    for p in segmentation_model.parameters()
    if p.requires_grad
)

print("Clean U-Net created successfully.")

print("\nDevice:", device)

print(
    "Total parameters:",
    f"{total_parameters:,}"
)

print(
    "Trainable parameters:",
    f"{trainable_parameters:,}"
)

# Test forward pass
sample_images, sample_masks = next(
    iter(train_loader)
)

sample_images = sample_images.to(device)

with torch.no_grad():
    sample_output = segmentation_model(
        sample_images
    )

print("\nInput shape:", sample_images.shape)
print("Output shape:", sample_output.shape)
print("Expected shape:", sample_masks.shape)

Clean U-Net created successfully.

Device: cpu
Total parameters: 483,441
Trainable parameters: 483,441

Input shape: torch.Size([8, 3, 224, 224])
Output shape: torch.Size([8, 1, 224, 224])
Expected shape: torch.Size([8, 1, 224, 224])


In [3]:
# ============================================================
# Step 24 - Loss Functions and Clean U-Net Training
# ============================================================

import torch
import torch.nn as nn
import time
import pandas as pd


# ------------------------------------------------------------
# Dice Loss
# ------------------------------------------------------------

class DiceLoss(nn.Module):

    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):

        probabilities = torch.sigmoid(logits)

        probabilities = probabilities.reshape(
            probabilities.size(0), -1
        )

        targets = targets.reshape(
            targets.size(0), -1
        )

        intersection = (
            probabilities * targets
        ).sum(dim=1)

        dice = (
            2.0 * intersection + self.smooth
        ) / (
            probabilities.sum(dim=1)
            + targets.sum(dim=1)
            + self.smooth
        )

        return 1.0 - dice.mean()


# ------------------------------------------------------------
# Loss functions
# ------------------------------------------------------------

bce_loss = nn.BCEWithLogitsLoss()
dice_loss = DiceLoss()


def combined_loss(logits, targets):

    bce = bce_loss(logits, targets)
    dice = dice_loss(logits, targets)

    return bce + dice


# ------------------------------------------------------------
# Training configuration
# ------------------------------------------------------------

EPOCHS = 10
LEARNING_RATE = 1e-3

optimizer = torch.optim.Adam(
    segmentation_model.parameters(),
    lr=LEARNING_RATE
)

print("Training configuration")
print("=" * 50)
print("Epochs:", EPOCHS)
print("Learning rate:", LEARNING_RATE)
print("Device:", device)


# ------------------------------------------------------------
# Validation function
# ------------------------------------------------------------

def validate_loss(model, loader):

    model.eval()

    total_loss = 0.0
    total_batches = 0

    with torch.no_grad():

        for images, masks in loader:

            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)

            loss = combined_loss(
                outputs,
                masks
            )

            total_loss += loss.item()
            total_batches += 1

    return total_loss / max(total_batches, 1)


# ------------------------------------------------------------
# Training
# ------------------------------------------------------------

training_history = []

best_val_loss = float("inf")
best_epoch = 0

print("\nStarting clean U-Net training...\n")

for epoch in range(EPOCHS):

    start_time = time.time()

    segmentation_model.train()

    running_loss = 0.0
    batches = 0

    for images, masks in train_loader:

        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()

        outputs = segmentation_model(images)

        loss = combined_loss(
            outputs,
            masks
        )

        loss.backward()

        optimizer.step()

        running_loss += loss.item()
        batches += 1

    train_loss = (
        running_loss / max(batches, 1)
    )

    val_loss = validate_loss(
        segmentation_model,
        val_loader
    )

    elapsed = time.time() - start_time

    # --------------------------------------------------------
    # Save best weights
    # --------------------------------------------------------

    if val_loss < best_val_loss:

        best_val_loss = val_loss
        best_epoch = epoch + 1

        torch.save(
            segmentation_model.state_dict(),
            "best_clean_unet.pth"
        )

        best_marker = " <-- BEST"

    else:
        best_marker = ""

    training_history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss
    })

    print(
        f"Epoch [{epoch + 1}/{EPOCHS}] "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Time: {elapsed:.1f}s"
        f"{best_marker}"
    )


print("\nTraining completed successfully.")

print("\nBest epoch:", best_epoch)
print("Best validation loss:", f"{best_val_loss:.4f}")

print("\nBest model saved:")
print("best_clean_unet.pth")

# Convert history to DataFrame
training_history_df = pd.DataFrame(
    training_history
)

training_history_df

Training configuration
Epochs: 10
Learning rate: 0.001
Device: cpu

Starting clean U-Net training...

Epoch [1/10] Train Loss: 1.1879 | Val Loss: 1.2322 | Time: 53.0s <-- BEST
Epoch [2/10] Train Loss: 1.0033 | Val Loss: 0.9368 | Time: 52.3s <-- BEST
Epoch [3/10] Train Loss: 0.8939 | Val Loss: 0.9616 | Time: 97.5s
Epoch [4/10] Train Loss: 0.8032 | Val Loss: 1.0848 | Time: 156.9s
Epoch [5/10] Train Loss: 0.7460 | Val Loss: 0.8318 | Time: 156.6s <-- BEST
Epoch [6/10] Train Loss: 0.6969 | Val Loss: 0.7226 | Time: 157.2s <-- BEST
Epoch [7/10] Train Loss: 0.6579 | Val Loss: 0.7733 | Time: 157.1s
Epoch [8/10] Train Loss: 0.6216 | Val Loss: 0.6171 | Time: 152.1s <-- BEST
Epoch [9/10] Train Loss: 0.5935 | Val Loss: 0.7416 | Time: 158.2s
Epoch [10/10] Train Loss: 0.5707 | Val Loss: 0.6737 | Time: 166.8s

Training completed successfully.

Best epoch: 8
Best validation loss: 0.6171

Best model saved:
best_clean_unet.pth


,epoch,train_loss,val_loss
0,1,1.187941,1.232161
1,2,1.003307,0.936818
2,3,0.893948,0.961567
3,4,0.803236,1.084810
4,5,0.746034,0.831756
5,6,0.696904,0.722566
6,7,0.657912,0.773310
7,8,0.621561,0.617085
8,9,0.593538,0.741633
9,10,0.570746,0.673732


In [4]:
# ============================================================
# Step 25 - Evaluate Best U-Net on Test Set
# ============================================================

import numpy as np
import torch


# ------------------------------------------------------------
# Load best model
# ------------------------------------------------------------

best_model_path = Path("best_clean_unet.pth")

if not best_model_path.exists():
    raise FileNotFoundError(
        f"Best model not found: {best_model_path.resolve()}"
    )

segmentation_model.load_state_dict(
    torch.load(
        best_model_path,
        map_location=device
    )
)

segmentation_model.eval()

print("Best U-Net loaded successfully.")
print("Best epoch:", best_epoch)
print("Best validation loss:", f"{best_val_loss:.4f}")


# ------------------------------------------------------------
# Dice Score
# ------------------------------------------------------------

def calculate_dice(predictions, targets, smooth=1e-7):

    predictions = predictions.reshape(
        predictions.shape[0], -1
    )

    targets = targets.reshape(
        targets.shape[0], -1
    )

    intersection = (
        predictions * targets
    ).sum(dim=1)

    dice = (
        2.0 * intersection + smooth
    ) / (
        predictions.sum(dim=1)
        + targets.sum(dim=1)
        + smooth
    )

    return dice.mean().item()


# ------------------------------------------------------------
# IoU Score
# ------------------------------------------------------------

def calculate_iou(predictions, targets, smooth=1e-7):

    predictions = predictions.reshape(
        predictions.shape[0], -1
    )

    targets = targets.reshape(
        targets.shape[0], -1
    )

    intersection = (
        predictions * targets
    ).sum(dim=1)

    union = (
        predictions
        + targets
        - predictions * targets
    ).sum(dim=1)

    iou = (
        intersection + smooth
    ) / (
        union + smooth
    )

    return iou.mean().item()


# ------------------------------------------------------------
# Test evaluation
# ------------------------------------------------------------

test_dice_scores = []
test_iou_scores = []

with torch.no_grad():

    for images, masks in test_loader:

        images = images.to(device)
        masks = masks.to(device)

        outputs = segmentation_model(images)

        probabilities = torch.sigmoid(outputs)

        predictions = (
            probabilities >= 0.5
        ).float()

        dice = calculate_dice(
            predictions,
            masks
        )

        iou = calculate_iou(
            predictions,
            masks
        )

        test_dice_scores.append(dice)
        test_iou_scores.append(iou)


# ------------------------------------------------------------
# Final metrics
# ------------------------------------------------------------

final_test_dice = float(
    np.mean(test_dice_scores)
)

final_test_iou = float(
    np.mean(test_iou_scores)
)


print("\nFinal Clean Test Results")
print("=" * 55)

print(
    f"Dice Score : {final_test_dice:.4f}"
)

print(
    f"IoU Score  : {final_test_iou:.4f}"
)

print("\nTest images:", len(test_df))
print("Test cases :", test_df["case_number"].nunique())

Best U-Net loaded successfully.
Best epoch: 8
Best validation loss: 0.6171

Final Clean Test Results
Dice Score : 0.5490
IoU Score  : 0.4107

Test images: 43
Test cases : 38


In [5]:
# ============================================================
# Step 26 - Save Final Clean Ultrasound Model
# ============================================================

import json
import shutil
from pathlib import Path

# ------------------------------------------------------------
# Final model directory
# ------------------------------------------------------------

FINAL_IMAGE_MODEL_DIR = (
    PROJECT_ROOT
    / "models"
    / "image"
)

FINAL_IMAGE_MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# Final model path
# ------------------------------------------------------------

FINAL_MODEL_PATH = (
    FINAL_IMAGE_MODEL_DIR
    / "ultrasound_unet_clean.pth"
)

FINAL_METADATA_PATH = (
    FINAL_IMAGE_MODEL_DIR
    / "ultrasound_unet_clean_info.json"
)

# ------------------------------------------------------------
# Save best clean model directly
# ------------------------------------------------------------

torch.save(
    segmentation_model.state_dict(),
    FINAL_MODEL_PATH
)

# ------------------------------------------------------------
# Save metadata
# ------------------------------------------------------------

final_metadata = {
    "model_name": "UNetSmall",
    "model_type": "Binary Ultrasound Segmentation",
    "input_channels": 3,
    "output_channels": 1,
    "input_size": [224, 224],

    "epochs": int(EPOCHS),
    "learning_rate": float(LEARNING_RATE),
    "batch_size": int(BATCH_SIZE),

    "best_epoch": int(best_epoch),
    "best_validation_loss": float(best_val_loss),

    "training_images": int(len(train_df)),
    "validation_images": int(len(val_df)),
    "test_images": int(len(test_df)),

    "training_cases": int(
        train_df["case_number"].nunique()
    ),
    "validation_cases": int(
        val_df["case_number"].nunique()
    ),
    "test_cases": int(
        test_df["case_number"].nunique()
    ),

    "test_dice": float(final_test_dice),
    "test_iou": float(final_test_iou),

    "duplicate_removed_images": 11,
    "clean_unique_images": 455,

    "device": str(device)
}

with open(
    FINAL_METADATA_PATH,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        final_metadata,
        file,
        indent=4
    )

# ------------------------------------------------------------
# Verify
# ------------------------------------------------------------

print("Final clean ultrasound model saved successfully.\n")

print(
    "Model file:",
    FINAL_MODEL_PATH
)

print(
    "Model exists:",
    FINAL_MODEL_PATH.exists()
)

print(
    "\nMetadata file:",
    FINAL_METADATA_PATH
)

print(
    "Metadata exists:",
    FINAL_METADATA_PATH.exists()
)

print("\nFinal Test Metrics:")
print(
    f"Dice Score : {final_test_dice:.4f}"
)
print(
    f"IoU Score  : {final_test_iou:.4f}"
)

Final clean ultrasound model saved successfully.

Model file: D:\MultiModalCancerTherapy_Structure\MultiModalCancerTherapy_Structure\models\image\ultrasound_unet_clean.pth
Model exists: True

Metadata file: D:\MultiModalCancerTherapy_Structure\MultiModalCancerTherapy_Structure\models\image\ultrasound_unet_clean_info.json
Metadata exists: True

Final Test Metrics:
Dice Score : 0.5490
IoU Score  : 0.4107
